In [ ]:
# make sure to unzip the dataset under /data

In [ ]:
import numpy as np
import rasterio
from rasterio.transform import from_origin
import xarray as xr
import os
import glob
import datetime


files = glob.glob("data/baseline/*.nc4")
os.makedirs("output", exist_ok=True)
for file in files:
    ds = xr.open_dataset(file)
    
    filename = file.split("/")[-1]
    date = filename.split("_")[1]
    time = filename.split("_")[2][:-1]
    level = filename.split("_")[-1].split(".")[0]
    final = date + time 
    dt = datetime.datetime.strptime(final, "%Y%m%d%H%M").strftime("%Y-%m-%dT%H:%M:%S")
    cog_filename = f"GCHP_CONUS_O3_{level}_{dt}Z.tif"

    
    # Extract array and coordinates
    data = ds["SpeciesConcVV_O3"].squeeze().values  # shape (lat, lon)
    # Replace 2D lat/lon variables with 1D coordinates
    ds = ds.assign_coords({
        "lon": ds["lon"].values[0, :],   # 1D lon
        "lat": ds["lat"].values[:, 0]    # 1D lat
    })
    # Replace NaNs with -9999
    data_filled = np.where(np.isnan(data), -9999, data)
    # Get coordinates
    lat = ds["lat"].values
    lon = ds["lon"].values
    nlat, nlon = data.shape
    
    # Ensure latitude is ordered from north to south
    if lat[0] < lat[-1]:  # increasing → need to flip
        data_filled = np.flipud(data_filled)
        lat = lat[::-1]
    # # Define geotransform
    transform = from_origin(
        west=lon.min(),       # left
        north=lat.max(),      # top
        xsize=np.abs(lon[1] - lon[0]),  # lon resolution
        ysize=np.abs(lat[1] - lat[0])   # lat resolution
    )

    print(cog_filename, data_filled.shape)
    
    # Save as GeoTIFF
    with rasterio.open(
        f"output/{cog_filename}",
        "w",
        driver="GTiff",
        height=nlat,
        width=nlon,
        count=1,
        dtype=data_filled.dtype,
        crs="EPSG:4326",
        transform=transform,
        nodata=-9999,
    ) as dst:
        dst.write(data_filled, 1)








